# 课后练习解答（02.03_dataset_exploration）

本解答对应章节课后练习，共 15 题。

### 问题1（单选题）

**题目：** Tiny ImageNet 有 200 类，训练集每类 500 张、验证集每类 50 张，验证集总数是？
A. 10,000
B. 50,000
C. 100,000
D. 20,000

**解答：** A

**解析：** 200 × 50 = 10,000 张，训练集为 200 × 500 = 100,000 张。


### 问题2（单选题）

**题目：** torchvision.datasets.ImageFolder 的类别索引顺序取决于？
A. 子目录名称排序
B. val_annotations.txt
C. 文件系统返回顺序
D. 图片修改时间

**解答：** A

**解析：** ImageFolder 会按子目录名排序生成 class_to_idx，保证跨平台一致。


### 问题3（单选题）

**题目：** RandomResizedCrop(size=224, scale=(0.08, 1.0), ratio=(3/4, 4/3)) 中 scale 表示？
A. 裁剪区域面积占原图面积的比例范围
B. 输出尺寸缩放倍数
C. 亮度缩放范围
D. 通道数

**解答：** A

**解析：** scale 约束裁剪框面积相对原图面积的比例，ratio 约束裁剪框宽高比。


### 问题4（多选题）

**题目：** 以下哪些 transform 会改变输出图像的宽高？
A. Resize(256)
B. RandomResizedCrop(224)
C. CenterCrop(224)
D. Normalize

**解答：** ABC

**解析：** Normalize 只做逐通道标准化，不改变空间尺寸。


### 问题5（多选题）

**题目：** 关于 RandAugment(num_ops=2, magnitude=9) 的正确说法包括？
A. 每个样本从操作集中随机选择 2 个增强操作
B. 每个样本使用完全相同的 2 个操作
C. magnitude 控制增强强度
D. 可包含旋转、平移、对比度等图像级操作

**解答：** ACD

**解析：** RandAugment 每次随机采样操作组合，magnitude 统一控制强度。


### 问题6（判断题）

**题目：** Tiny ImageNet 原始图片尺寸为 224×224。

**解答：** 错

**解析：** Tiny ImageNet 原始图片为 64×64，需要 Resize/Crop 到 224。


### 问题7（判断题）

**题目：** RandomErasing 通常只用于训练集，不应默认加入验证集。

**解答：** 对

**解析：** 验证集应模拟稳定分布，随机遮挡会引入噪声，降低指标可比性。


### 问题8（填空题）

**题目：** 验证集按 class 子目录整理后，每类图片数为 ____，验证集总数为 ____。

**解答：** 50；10,000


### 问题9（填空题）

**题目：** transforms.ToTensor() 将像素值缩放到 ____ 区间；transforms.Normalize 使用 ____ 和标准差做标准化。

**解答：** [0,1]；均值


### 问题10（简答题）

**题目：** 为什么 64×64 的 Tiny ImageNet 通常先 Resize 到 256，再做 CenterCrop 到 224，而不是直接 Resize 到 224？

**解答：** 直接放大到目标尺寸容易产生锯齿和边缘伪影；先放大到略大于目标尺寸再居中裁剪，能保留更多有效区域，并让后续卷积更少受边界效应影响。


### 问题11（简答题）

**题目：** 增强强度过大为什么可能降低准确率？请结合语义破坏与分布漂移说明。

**解答：** 过强增强可能把类别关键区域裁掉、颜色/几何变化超出真实分布，使模型学到错误映射；同时训练分布与验证分布差距拉大，造成训练指标好但验证指标差。


### 问题12（代码设计题）

**题目：** 编写 create_transforms(train)，训练时使用 RandomResizedCrop(224)、RandAugment、RandomErasing、ToTensor、Normalize；验证时使用 Resize(256)+CenterCrop(224)+ToTensor+Normalize，并说明验证集为何不用随机增强。

**解答：** ```python
from torchvision import transforms

def create_transforms(train=True):
    normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    if train:
        return transforms.Compose([
            transforms.RandomResizedCrop(224, scale=(0.08, 1.0), ratio=(3/4, 4/3)),
            transforms.RandAugment(num_ops=2, magnitude=9),
            transforms.ToTensor(),
            transforms.RandomErasing(p=0.25),
            normalize,
        ])
    return transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        normalize,
    ])
```
验证集必须保持确定性，随机增强会改变每次评估输入，导致指标不可复现且无法横向比较。


### 问题13（单选题）

**题目：** 训练集与验证集类别分布差异很大时，最可能的影响是？
A. 验证指标无法真实反映模型泛化能力
B. 训练速度变慢
C. 显存升高
D. 学习率必须调大

**解答：** A

**解析：** 分布不一致使验证集失去代表性，指标会系统性偏高或偏低。


### 问题14（多选题）

**题目：** 验证集准备与评估中必须保持一致的是？
A. 与训练相同的 Normalize 参数
B. model.eval() 与 torch.no_grad()
C. 输入尺寸 224×224
D. RandAugment 随机增强

**解答：** ABC

**解析：** 验证输入需要与训练同分布，但不应引入随机增强。


### 问题15（简答题）

**题目：** 设计一种方法快速检查数据增强是否正确：既要看到增强效果，又要确认标签与图片对应关系。

**解答：** 固定随机种子后对同一 batch 做多次可视化，使用 torchvision.utils.make_grid 输出增强前后图像；同时打印每张图对应的 class name，人工核对类别内容未被破坏，并检查 tensor 的 mean/std 与 Normalize 参数是否匹配。
